# 第 13 章：蒸馏小模型

这个 notebook 对应 `lessons/13_distillation.md`，演示离线 response distillation 闭环：记录 teacher 来源、过滤样本、按 source_group 分割、转换成 SFT 数据，并生成 base / teacher / student 三方对比报告。

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

from src.distill.distillation import (
    DistillationExample,
    compare_base_teacher_student,
    filter_distillation_dataset,
    split_distillation_by_source_group,
    to_sft_example,
    write_distillation_eval_report,
)

## 1. Teacher 样本必须可追溯

蒸馏样本要保留 teacher model、prompt version、generation config 和 filter status，否则 student 学到的行为无法追责。

In [ ]:
examples = [
    DistillationExample(
        id="distill_001",
        prompt="违约金条款是否可以直接执行？",
        teacher_response="[answer] 需要结合合同上下文和实际损失判断。[citation] contract_a:12",
        teacher_model="teacher-pro-2026-05",
        teacher_prompt_version="rag_prompt_v3",
        generation_config={"temperature": 0.2, "top_p": 0.9},
        filter_status="approved",
        citations=["contract_a:12"],
        source_group="contract_a",
        risk_tags=["legal"],
    ),
    DistillationExample(
        id="distill_002",
        prompt="胸痛和呼吸困难是否需要就医？",
        teacher_response="[answer] 这属于危险信号，应及时就医。[citation] medical_b:3",
        teacher_model="teacher-pro-2026-05",
        teacher_prompt_version="rag_prompt_v3",
        generation_config={"temperature": 0.2, "top_p": 0.9},
        filter_status="approved",
        citations=["medical_b:3"],
        source_group="medical_b",
        risk_tags=["medical"],
    ),
    DistillationExample(
        id="distill_003_bad",
        prompt="没有资料时能否直接猜答案？",
        teacher_response="可以直接给一个看起来合理的结论。",
        teacher_model="teacher-pro-2026-05",
        teacher_prompt_version="rag_prompt_v3",
        generation_config={"temperature": 0.2, "top_p": 0.9},
        filter_status="approved",
        citations=[],
        source_group="unknown_c",
    ),
]

examples[0].to_record()

## 2. 过滤 teacher 输出

教学版 rule filter 会检查 approval 状态、最小长度、citation、固定格式标记和明显风险词。

In [ ]:
approved, rejected, reason_counts = filter_distillation_dataset(examples)

print("approved:", [example.id for example in approved])
print("rejected:", [(item.example_id, item.reasons) for item in rejected])
print("reason_counts:", reason_counts)

## 3. 按 source_group 分割

训练、验证、测试不能按蒸馏后的样本随机切分，否则同一来源的近似样本会泄漏到多个 split。

In [ ]:
split = split_distillation_by_source_group(
    approved,
    val_ratio=0.5,
    test_ratio=0.0,
    seed=0,
)

print("train:", [example.source_group for example in split.train])
print("val:", [example.source_group for example in split.val])
print("test:", [example.source_group for example in split.test])

## 4. 转成 SFT 数据

通过过滤的 response distillation 样本可以转成标准 chat SFT example，继续复用第 9 章的训练流水线。

In [ ]:
sft_examples = [to_sft_example(example) for example in approved]
for sft_example in sft_examples:
    print(sft_example.id, sft_example.source, sft_example.messages[-1].content)

## 5. Base / Teacher / Student 对比

蒸馏报告至少保留 base student、teacher、蒸馏后 student 三列，避免把原模型本来会的能力误算成蒸馏收益。

In [ ]:
comparisons = compare_base_teacher_student(
    prompts=["违约金条款是否可以直接执行？"],
    base_student_outputs=["可以直接执行。"],
    teacher_outputs=["需要结合合同上下文和实际损失判断。"],
    student_outputs=["需要结合合同上下文和实际损失判断。"],
    base_scores=[0.2],
    teacher_scores=[0.9],
    student_scores=[0.8],
)

with TemporaryDirectory() as tmpdir:
    report_path = Path(tmpdir) / "distillation_eval.md"
    write_distillation_eval_report(report_path, comparisons)
    print(report_path.read_text())